# Broadcast Communication

By default, communication occurs between all satellites that are specified by the communication
method. This tutorial shows how to use two classes to configure a broadcast action that must be
taken for communication to occur:

* [Broadcast](../api_reference/act/index.html#bsk_rl.act.Broadcast), which gives satellites an action
  that enables communication from them at the end of the current step.
* [BroadcastCommunication](../api_reference/comm/index.html#bsk_rl.comm.BroadcastCommunication), which
  can be combined with another communication method to limit communication from broadcasters to
  those satellites satisfying the requirements of the other communication method.

## Configuring the Environment

For this example, a multisatellite target imaging environment will be used. The goal is
to maximize the value of unique images taken. This configuration is similar to the 
[Multi-Agent Environments](../examples/multiagent_envs.html) example.

In [1]:
from typing import ClassVar

from bsk_rl import act, comm, data, obs, sats, scene
from bsk_rl.sim import dyn, fsw


class ImagingSatellite(sats.ImagingSatellite):
    observation_spec: ClassVar[list[obs.Observation]] = [
        obs.OpportunityProperties(
            dict(prop="priority"),
            dict(prop="opportunity_open", norm=5700.0),
            n_ahead_observe=4,
        )
    ]
    action_spec: ClassVar[list[act.Action]] = [
        act.Broadcast(duration=15.0),
        act.Image(n_ahead_image=4),
    ]
    dyn_type = dyn.FullFeaturedDynModel
    fsw_type = fsw.SteeringImagerFSWModel


ACTION_BROADCAST = 0
ACTION_IMAGE_0 = 1
ACTION_IMAGE_1 = 2
ACTION_IMAGE_2 = 3
ACTION_IMAGE_3 = 4

Satellite properties are set to give the satellite near-unlimited power and storage resources. To randomize some parameters in a correlated manner across satellites, a ``sat_arg_randomizer`` is set and passed to the environment. In this case, the satellites are distributed in a trivial single-plane Walker-delta constellation.

In [2]:
from bsk_rl.utils.orbital import walker_delta_args

N_AGENTS = 2
sat_args = dict(
    imageAttErrorRequirement=0.01,
    imageRateErrorRequirement=0.01,
    batteryStorageCapacity=1e9,
    storedCharge_Init=1e9,
    dataStorageCapacity=1e12,
    u_max=0.4,
    K1=0.25,
    K3=3.0,
    omega_max=0.087,
    servo_Ki=5.0,
    servo_P=150 / 5,
)
sat_arg_randomizer = walker_delta_args(
    altitude=800.0, inc=60.0, n_planes=1, clustersize=N_AGENTS, clusterspacing=5.0
)

A communication type is defined that uses multidegree line-of-sight communication with broadcasting.

In [3]:
class BroadcastLOS(comm.BroadcastCommunication, comm.LOSMultiCommunication):
    pass

Finally, the environment can be instantiated.

In [4]:
from bsk_rl import ConstellationTasking

env = ConstellationTasking(
    satellites=[ImagingSatellite(f"EO_{i + 1}", sat_args) for i in range(N_AGENTS)],
    scenario=scene.UniformTargets(1000),
    rewarder=data.UniqueImageReward(),
    communicator=BroadcastLOS(),
    sat_arg_randomizer=sat_arg_randomizer,
    log_level="INFO",
)
_ = env.reset()

2026-11-04 04:33:05,035 gym                            INFO       Resetting environment with seed=535784546


2026-11-04 04:33:05,036 scene.targets                  INFO       Generating 1000 targets


2026-11-04 04:33:05,084 sats.satellite.EO_1            INFO       <0.00> EO_1: Finding opportunity windows from 0.00 to 600.00 seconds


2026-11-04 04:33:05,101 sats.satellite.EO_2            INFO       <0.00> EO_2: Finding opportunity windows from 0.00 to 600.00 seconds


2026-11-04 04:33:05,117 gym                            INFO       <0.00> Environment reset


In the first step, both agents are tasked with an imaging action, and one successfully images a target.

In [5]:
_ = env.step({"EO_1": ACTION_IMAGE_3, "EO_2": ACTION_IMAGE_2})

2026-11-04 04:33:05,120 gym                            INFO       <0.00> === STARTING STEP ===


2026-11-04 04:33:05,120 sats.satellite.EO_1            INFO       <0.00> EO_1: target index 3 tasked


2026-11-04 04:33:05,121 sats.satellite.EO_1            INFO       <0.00> EO_1: Target(tgt-453) tasked for imaging


2026-11-04 04:33:05,121 sats.satellite.EO_1            INFO       <0.00> EO_1: Target(tgt-453) window enabled: 0.0 to 166.5


2026-11-04 04:33:05,121 sats.satellite.EO_1            INFO       <0.00> EO_1: setting timed terminal event at 166.5


2026-11-04 04:33:05,122 sats.satellite.EO_2            INFO       <0.00> EO_2: target index 2 tasked


... (3 outputs left out)

2026-11-04 04:33:05,131 sats.satellite.EO_2            INFO       <57.00> EO_2: imaged Target(tgt-453)


2026-11-04 04:33:05,132 data.base                      INFO       <57.00> Total reward: {'EO_2': 0.6470600531868214}


2026-11-04 04:33:05,132 sats.satellite.EO_2            INFO       <57.00> EO_2: Satellite EO_2 requires retasking


2026-11-04 04:33:05,134 gym                            INFO       <57.00> Step reward: {'EO_2': 0.6470600531868214}


When both select the broadcast action, data is shared in both directions. This is subject to line-of-sight availability as well as selecting the correct action.

In [6]:
_ = env.step({"EO_1": ACTION_BROADCAST, "EO_2": ACTION_BROADCAST})

2026-11-04 04:33:05,136 gym                            INFO       <57.00> === STARTING STEP ===


2026-11-04 04:33:05,137 sats.satellite.EO_1            INFO       <57.00> EO_1: action_broadcast tasked for 15.0 seconds


2026-11-04 04:33:05,137 sats.satellite.EO_1            INFO       <57.00> EO_1: setting timed terminal event at 72.0


2026-11-04 04:33:05,138 sats.satellite.EO_2            INFO       <57.00> EO_2: action_broadcast tasked for 15.0 seconds


2026-11-04 04:33:05,138 sats.satellite.EO_2            INFO       <57.00> EO_2: setting timed terminal event at 72.0


2026-11-04 04:33:05,140 sats.satellite.EO_1            INFO       <72.00> EO_1: timed termination at 72.0 for broadcast


... (3 outputs left out)

2026-11-04 04:33:05,142 comm.communication             INFO       <72.00> Optimizing data communication between all pairs of satellites


2026-11-04 04:33:05,142 sats.satellite.EO_1            INFO       <72.00> EO_1: Satellite EO_1 requires retasking


2026-11-04 04:33:05,142 sats.satellite.EO_2            INFO       <72.00> EO_2: Satellite EO_2 requires retasking


2026-11-04 04:33:05,144 gym                            INFO       <72.00> Step reward: {}


One agent can broadcast while the others are completing a different task.

In [7]:
_ = env.step({"EO_1": ACTION_IMAGE_2, "EO_2": ACTION_BROADCAST})  # Agent 1 broadcasts

2026-11-04 04:33:05,147 gym                            INFO       <72.00> === STARTING STEP ===


2026-11-04 04:33:05,147 sats.satellite.EO_1            INFO       <72.00> EO_1: target index 2 tasked


2026-11-04 04:33:05,147 sats.satellite.EO_1            INFO       <72.00> EO_1: Target(tgt-549) tasked for imaging


2026-11-04 04:33:05,148 sats.satellite.EO_1            INFO       <72.00> EO_1: Target(tgt-549) window enabled: 0.0 to 177.7


2026-11-04 04:33:05,148 sats.satellite.EO_1            INFO       <72.00> EO_1: setting timed terminal event at 177.7


2026-11-04 04:33:05,148 sats.satellite.EO_2            INFO       <72.00> EO_2: action_broadcast tasked for 15.0 seconds


... (2 outputs left out)

2026-11-04 04:33:05,152 data.base                      INFO       <87.00> Total reward: {}


2026-11-04 04:33:05,152 comm.communication             INFO       <87.00> Communicating data in 1 direction.


2026-11-04 04:33:05,152 sats.satellite.EO_2            INFO       <87.00> EO_2: Satellite EO_2 requires retasking


2026-11-04 04:33:05,154 gym                            INFO       <87.00> Step reward: {}
